In [ ]:
#parameters
datadir = ""
scan_number = ""
ramp_x1 = ""
ramp_x2 = ""
ramp_y1 = ""
ramp_y2 = ""
crop = 0

In [ ]:
import os
import h5py
import logging
import numpy as np
import matplotlib.pyplot as plt
from imaging_toolbox.ptychography import remove_ramp_and_unwrap_phase

In [ ]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

In [ ]:
if not os.path.isdir(datadir):
    raise ValueError("data directory not found")

In [ ]:
logger.debug(
    f"""
    Provided arguments:
    Scan Number: {scan_number}
        ramp_x1: {ramp_x1}
        ramp_x2: {ramp_x2}
        ramp_y1: {ramp_y1}
        ramp_y2: {ramp_y2}
           crop: {crop}
    """
)

In [ ]:
ramp_x1 = None if ramp_x1.lower() == "none" else int(ramp_x1)
ramp_x2 = None if ramp_x2.lower() == "none" else int(ramp_x2)
ramp_y1 = None if ramp_y1.lower() == "none" else int(ramp_y1)
ramp_y2 = None if ramp_y2.lower() == "none" else int(ramp_y2)

In [ ]:
fpath = f"{datadir}/processed/ptychography/scan_{scan_number}/scan_{scan_number}.ptyr"

In [ ]:
with h5py.File(fpath, "r") as f:
    data = np.array(f["content/obj/Sscan_00G00/data"][0,crop:-crop,crop:-crop])
    phase = np.angle(data)

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2)
ax1.imshow(phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax1.set_title("Phase Ramp ROI")
ax1.axis("off")
ax1.axvline(ramp_x1,color='r')
ax1.axvline(ramp_x2,color='r')
ax1.axhline(ramp_y1,color='b')
ax1.axhline(ramp_y2,color='b')

rm_mask = np.zeros_like(phase, dtype="bool")
rm_mask[ramp_y1:ramp_y2, ramp_x1:ramp_x2] = True
ramp_corr_phase = remove_ramp_and_unwrap_phase(phase, mask=rm_mask)

ax2.imshow(ramp_corr_phase, cmap="bone", vmin=-np.pi, vmax=np.pi)
ax2.set_title("Corrected")
ax2.axis("off")

plt.show()